# 15 N-glycan Structural Classification

## Purpose

This notebook builds a separate rule-based workflow that classifies compact IUPAC glycan sequences by structure rather than depending only on the existing label table.

The main goals are:

- detect likely `N-glycan` rows from the sequence itself
- subclass structural N-glycans into `High mannose`, `Hybrid`, `Complex`, or `Paucimannose/truncated` when the rule evidence is clear
- keep an explicit `Unresolved N-glycan` outcome instead of forcing weak assignments
- evaluate how these structural calls compare with the current project labels

This notebook is meant to complement notebook `14`, not replace it. Notebook `14` asks whether saved embedding spaces linearly separate an N-glycan task derived from the current labels. This notebook asks an earlier question: how much of the corpus can we classify from compact IUPAC structure alone, and where do those structural calls agree or disagree with the current labels?


## User settings

This is the main cell to review before running the notebook.

**What this cell does**
- defines the Google Drive project location
- defines the repository settings used during runtime setup
- defines the prepared classification input filenames
- defines the structural-classification run label and overwrite policy

**Why this cell is being run**
- to keep the notebook-specific editable values in one obvious place near the top of the notebook

**Expected output**
- the resolved project root
- the resolved classification-prep directory
- the structural run label and overwrite setting for this run

**How to interpret the output**
- if any path looks wrong here, fix it before running the later cells
- if the run label is unclear, update it now so the saved outputs are easy to recognize later


In [ ]:
from pathlib import Path

# Update PROJECT_ROOT if your Google Drive project folder uses a different
# name or location.
PROJECT_ROOT = Path('/content/drive/MyDrive/ProjectRoot')

# These repository settings are used only during the Colab runtime setup.
GITHUB_OWNER = 'hb791-dev'
REPO_NAME = 'glycan-roberta'
GITHUB_REF = 'main'
REPO_DIR = Path('/content') / REPO_NAME

# Notebook-09 writes the prepared classification tables that this notebook
# audits with structure-based N-glycan rules.
CLASSIFICATION_PREP_SUBDIR = Path('results') / 'classification_prep'
TRAIN_CLASSIFICATION_FILENAME = 'train_classification.csv'
VAL_CLASSIFICATION_FILENAME = 'val_classification.csv'
TEST_CLASSIFICATION_FILENAME = 'test_classification.csv'

# Choose which standard splits to include in this structural audit.
SPLITS_TO_INCLUDE = ('train', 'val', 'test')

# Use a descriptive run label so the saved output folder is easy to find.
STRUCTURAL_RUN_LABEL = 'compact_iupac_structural_rules_v1'

# If True, the notebook may replace existing saved outputs for this run.
# If False, the helper will stop before overwriting files.
OVERWRITE_EXISTING_OUTPUTS = True

# These settings limit how many disagreement and unresolved rows are saved
# for manual review.
DISAGREEMENT_LIMIT = 200
UNRESOLVED_LIMIT = 200
DISPLAY_EXAMPLE_ROWS = 40

classification_prep_dir = PROJECT_ROOT / CLASSIFICATION_PREP_SUBDIR

print(f'Project root: {PROJECT_ROOT}')
print(f'Classification prep dir: {classification_prep_dir}')
print(f'Repository directory: {REPO_DIR}')
print(f'Splits to include: {SPLITS_TO_INCLUDE}')
print(f'Structural run label: {STRUCTURAL_RUN_LABEL}')
print(f'Overwrite existing outputs: {OVERWRITE_EXISTING_OUTPUTS}')


## Runtime setup

This cell prepares the Colab environment for the notebook. It mounts Google Drive, synchronizes the GitHub repository, and makes the project `src` code available for import.

We keep the setup code explicit here because the notebook must first download the repository before it can import the shared helper modules.

**What this cell does**
- mounts Google Drive
- clones the repository if needed
- fast-forward pulls the selected branch
- adds the repository root to the Python import path

**Why this cell is being run**
- to make sure the notebook is using the current shared helper code from GitHub

**Expected output**
- confirmation that Google Drive is mounted
- confirmation that the GitHub repository is available locally
- confirmation of the active repository directory

**How to interpret the output**
- if repository sync fails here, later imports from `src` will fail or use stale code
- if the repository directory is unexpected, the notebook may not be importing the intended project version


In [ ]:
# Standard library imports used for environment setup.
import subprocess
import sys

from google.colab import drive

# Mount Google Drive so the notebook can read the prepared classification
# tables and save the structural-classification outputs.
drive.mount('/content/drive')

# This structural workflow depends only on lightweight table handling.
!pip install -q pandas

repo_url = f'https://github.com/{GITHUB_OWNER}/{REPO_NAME}.git'

# Clone the repository into the Colab runtime the first time the notebook runs.
# If the repository is already present, pull the latest changes so the notebook
# uses the current helper code.
if not REPO_DIR.exists():
    print(f'Cloning repository from {repo_url} ...')
    subprocess.run(['git', 'clone', '--quiet', repo_url, str(REPO_DIR)], check=True)
else:
    print(f'Repository already exists at {REPO_DIR}.')

print(f"Updating repository to the latest '{GITHUB_REF}' changes...")
subprocess.run(
    ['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', GITHUB_REF],
    check=True,
)

# Add the repository root to the Python import path so the notebook can import
# shared helper modules from the src package.
repo_dir_str = str(REPO_DIR)
if repo_dir_str not in sys.path:
    sys.path.insert(0, repo_dir_str)

print(f'Repository directory: {REPO_DIR}')


## Import shared helpers

This notebook keeps the structural classifier and the evaluation summaries in `src/n_glycan_structural_classification.py` so the notebook body can stay focused on run selection and interpretation.

**What this cell does**
- imports the notebook-facing helper functions for structural classification
- reloads the helper module so local edits are picked up in the current runtime

**Why this cell is being run**
- to keep the repeatable logic in `src/` instead of duplicating it inline in the notebook

**Expected output**
- no printed output if the imports succeed

**How to interpret the output**
- if this cell runs quietly, the notebook is ready to build paths and run the structural audit
- if an import fails, fix the runtime setup step before editing later analysis cells


In [ ]:
# Import standard analysis tools used for notebook display and path handling.
import importlib

import pandas as pd
from IPython.display import display

# Reload the structural helper so the runtime reflects the latest staged edits.
import src.n_glycan_structural_classification as structural_helper
importlib.reload(structural_helper)

# Import the notebook-facing helper functions used throughout this workflow.
from src.n_glycan_structural_classification import (
    build_structural_classification_output_paths,
    build_structural_classification_run_config,
    run_structural_classification_workflow,
    save_structural_classification_run_config,
)


## Build output paths and save the run config

This step creates the standard output folder for the structural workflow and saves the editable notebook settings as JSON so later review does not depend on reopening the notebook.

**What this cell does**
- builds the standard results paths for this notebook run
- saves the active notebook settings to a JSON file

**Why this cell is being run**
- to make the run auditable and to keep saved outputs organized under one run label

**Expected output**
- the main output directory for this structural audit
- the saved config JSON path

**How to interpret the output**
- if the output directory is not what you expected, update `STRUCTURAL_RUN_LABEL` before continuing
- if the config path prints correctly, later review can recover the exact settings used for this run


In [ ]:
# Build the standard result paths so every saved table lands in one run folder.
output_paths = build_structural_classification_output_paths(
    PROJECT_ROOT,
    run_label=STRUCTURAL_RUN_LABEL,
)

# Save the editable notebook settings so the run can be reviewed later
# without reopening the notebook itself.
run_config = build_structural_classification_run_config(
    project_root=PROJECT_ROOT,
    classification_prep_dir=classification_prep_dir,
    splits_to_include=SPLITS_TO_INCLUDE,
    structural_run_label=STRUCTURAL_RUN_LABEL,
    overwrite_existing_outputs=OVERWRITE_EXISTING_OUTPUTS,
    disagreement_limit=DISAGREEMENT_LIMIT,
    unresolved_limit=UNRESOLVED_LIMIT,
    output_paths=output_paths,
)
save_structural_classification_run_config(output_paths['run_config_path'], run_config)

print(f"Output dir: {output_paths['results_dir']}")
print(f"Saved config: {output_paths['run_config_path']}")


## Run the structural classifier and evaluation workflow

This step loads the prepared classification tables from notebook `09`, annotates the current label view, applies the rule-based structural classifier to every compact IUPAC sequence, and saves the main audit tables.

**What this cell does**
- loads the train, validation, and test classification tables
- applies the structural N-glycan rule engine to each row
- builds structural class summaries and agreement tables
- saves the annotated rows and review tables as CSV files

**Why this cell is being run**
- to generate the core structural-classification outputs that the later review cells interpret

**Expected output**
- the number of rows processed
- the main saved CSV paths for this run

**How to interpret the output**
- a successful run here means the compact IUPAC parser and structural rule workflow were able to process the requested splits
- if this cell fails, the error message usually points to a path problem, an overwrite-policy problem, or a sequence pattern the parser does not yet support


In [ ]:
# Resolve the three prepared classification input files written by notebook 09.
train_classification_path = classification_prep_dir / TRAIN_CLASSIFICATION_FILENAME
val_classification_path = classification_prep_dir / VAL_CLASSIFICATION_FILENAME
test_classification_path = classification_prep_dir / TEST_CLASSIFICATION_FILENAME

# Run the full structural audit and keep the returned tables in memory for
# the review cells below.
workflow_results = run_structural_classification_workflow(
    train_csv_path=train_classification_path,
    val_csv_path=val_classification_path,
    test_csv_path=test_classification_path,
    output_paths=output_paths,
    splits_to_include=SPLITS_TO_INCLUDE,
    overwrite_existing_outputs=OVERWRITE_EXISTING_OUTPUTS,
    disagreement_limit=DISAGREEMENT_LIMIT,
    unresolved_limit=UNRESOLVED_LIMIT,
)

# Unpack the returned dataframes into readable notebook variables.
annotated_df = workflow_results['annotated_df']
class_summary_df = workflow_results['class_summary_df']
reason_summary_df = workflow_results['reason_summary_df']
binary_agreement_df = workflow_results['binary_agreement_df']
subclass_agreement_df = workflow_results['subclass_agreement_df']
disagreement_examples_df = workflow_results['disagreement_examples_df']
unresolved_examples_df = workflow_results['unresolved_examples_df']

print(f'Rows processed: {len(annotated_df):,}')
print(f"Saved annotated rows: {output_paths['annotated_rows_path']}")
print(f"Saved structural class summary: {output_paths['class_summary_path']}")
print(f"Saved structural reason summary: {output_paths['reason_summary_path']}")
print(f"Saved binary agreement summary: {output_paths['binary_agreement_path']}")
print(f"Saved subclass agreement summary: {output_paths['subclass_agreement_path']}")


## Review coverage and agreement

These tables answer the main evaluation questions for the first structural pass.

**What this cell does**
- displays the split-by-class structural coverage table
- displays the structural assignment reason summary
- displays the binary agreement summary against the current broad labels
- displays the subclass agreement summary inside current N-glycan rows

**Why this cell is being run**
- to judge how broad, conservative, and label-consistent the first structural rule set is

**Expected output**
- four review tables shown directly in the notebook

**How to interpret the output**
- high coverage with few unresolved rows suggests the rules are broad enough to support downstream experiments
- many disagreements suggest the current labels and structural rules are describing different biological subsets
- many unresolved rows suggest the parser or subclass rules still need refinement before they should become a primary target source


In [ ]:
# Review how many rows from each split land in each structural class.
print('Structural class counts by split')
display(class_summary_df)

# Review why rows were assigned to each class and how confident those
# assignments were.
print('Structural assignment reasons')
display(reason_summary_df)

# Compare the structural N-glycan detector against the current broad labels.
print('Binary agreement with current broad labels')
display(binary_agreement_df)

# Compare structural subclass calls against the current label-derived
# subclass view within rows already labeled as N-glycan.
print('Subclass agreement inside current N-glycan rows')
display(subclass_agreement_df)


## Review disagreement and unresolved examples

These example tables are the most useful place to inspect whether the rules fail for sensible biological reasons or because the parser and subclass logic need work.

**What this cell does**
- shows example rows where the structural binary call disagrees with the current broad labels
- shows example rows where the structural workflow returned `Unresolved N-glycan`
- prints the saved CSV paths for the manual-review tables

**Why this cell is being run**
- to make the first pass actionable by surfacing concrete examples rather than only summary counts

**Expected output**
- two example tables shown directly in the notebook
- the saved CSV paths for those example tables

**How to interpret the output**
- disagreement rows help identify where the current labels and the structural rules are telling different stories
- unresolved rows show which sequence patterns still need new rules or a broader fallback category


In [ ]:
# Review a small preview of rows where the structural binary call and the
# current broad labels disagree.
print('Binary disagreement examples')
display(disagreement_examples_df.head(DISPLAY_EXAMPLE_ROWS))

# Review a small preview of rows where the structural workflow found an
# apparent N-glycan pattern but could not place it confidently.
print('Unresolved structural examples')
display(unresolved_examples_df.head(DISPLAY_EXAMPLE_ROWS))

print(f"Saved disagreement examples: {output_paths['disagreement_examples_path']}")
print(f"Saved unresolved examples: {output_paths['unresolved_examples_path']}")
